# Factor Analysis using the CAPM and Fama-French Factor models

The main idea in Factor Analysis is to take a set of observed returns and decompose it into a set of explanatory returns.

We'll follow _Asset Management_ (Ang 2014, Oxford University Press) Chapter 10 and analyze the returns of Berkshire Hathaway.

First, we'll need the returns of Berkshire Hathaway which are contained in `data/brka_d_ret.csv`. Read it in as follows:

In [1]:
import pandas as pd
import risk_kit as rk
import statsmodels.api as sm
import numpy as np

%load_ext autoreload
%autoreload 2

Objective: decompose BRK.A returns into explanatory return components and identify the key performance drivers.

In [2]:
brka_d = pd.read_csv('data/brka_d_ret.csv', parse_dates = True, index_col = 0)
brka_d.head()

,BRKA
DATE,
1990-01-02,-0.005764
1990-01-03,0.000000
1990-01-04,0.005797
1990-01-05,-0.005764
1990-01-08,0.000000


In [3]:
max(brka_d.index)

Timestamp('2018-12-31 00:00:00')

Next, we need to convert these to monthly returns. The simplest way to do so is by using the `.resample` method, which allows you to run an aggregation function on each group of returns in a time series. We'll give it the grouping rule of 'M' which means _monthly_ (consult the `pandas`) documentation for other codes)

We want to compound the returns, and we already have the `compound` function in our toolkit, so let's load that up now, and then apply it to the daily returns.

In [4]:
brka_m = brka_d.resample('ME').apply(rk.compound).to_period('M')
brka_m.head()

,BRKA
DATE,
1990-01,-0.140634
1990-02,-0.030852
1990-03,-0.069204
1990-04,-0.003717
1990-05,0.067164


In [5]:
brka_m.to_csv('data/brka_m.csv')

Next, we need to load the explanatory variables, which is the Fama-French monthly returns data set. Load that as follows:

In [6]:
fff = rk.get_fff_returns()
fff.head()

,Mkt-RF,SMB,HML,RF
1926-07,0.0296,-0.0230,-0.0287,0.0022
1926-08,0.0264,-0.0140,0.0419,0.0025
1926-09,0.0036,-0.0132,0.0001,0.0023
1926-10,-0.0324,0.0004,0.0051,0.0032
1926-11,0.0253,-0.0020,-0.0035,0.0031


Next, we need to decompose the observed BRKA 1990-May 2012 as in Ang(2014) into the portion that's due to the market and the rest that is not due to the market, using the CAPM as the explanatory model.

i.e.

$$ R_{brka,t} - R_{f,t} = \alpha + \beta(R_{mkt,t} - R_{f,t}) + \epsilon_t $$

We can use the `stats.api` for the linear regression as follows:

In [10]:
brka_excess = brka_m.loc['1990':'2012-05'] - fff.loc['1990':'2012-05', ['RF']].values
mkt_excess = fff.loc['1990':'2012-05', ['Mkt-RF']]
exp_var = mkt_excess.copy()
exp_var["Constant"] = 1
lm = sm.OLS(brka_excess, exp_var).fit()

In [30]:
lm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   BRKA   R-squared:                       0.154
Model:                            OLS   Adj. R-squared:                  0.150
Method:                 Least Squares   F-statistic:                     48.45
Date:                Wed, 18 Feb 2026   Prob (F-statistic):           2.62e-11
Time:                        13:01:12   Log-Likelihood:                 388.47
No. Observations:                 269   AIC:                            -772.9
Df Residuals:                     267   BIC:                            -765.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Mkt-RF         0.5402      0.078      6.961      0.000       0.387       0.693
Constant       0.0061      0.004      1.744      0.082      -0.001       0.013
==============================================================================
Omnibus:                       45.698   Durbin-Watson:                   2.079
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              102.573
Skew:                           0.825   Prob(JB):                     5.33e-23
Kurtosis:                       5.535   Cond. No.                         22.2
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### The CAPM benchmark interpretation

This implies that the CAPM benchmark consists of 46 cents in T-Bills and 54 cents in the market. i.e. each dollar in the Berkshire Hathaway portfolio is equivalent to 46 cents in T-Bills and 54 cents in the market. Relative to this, the Berkshire Hathaway is adding (i.e. has $\alpha$ of) 0.61% _(per month!)_ although the degree of statistica significance is not very high.

Now, let's add in some additional explanatory variables, namely Value and Size.

In [31]:
exp_var["Value"] = fff.loc["1990":"2012-05", ['HML']]
exp_var["Size"] = fff.loc["1990":"2012-05", ['SMB']]
exp_var.head()

,Mkt-RF,Constant,Value,Size
1990-01,-0.0785,1,0.0087,-0.0129
1990-02,0.0111,1,0.0061,0.0103
1990-03,0.0183,1,-0.0290,0.0152
1990-04,-0.0336,1,-0.0255,-0.0050
1990-05,0.0842,1,-0.0374,-0.0257


In [32]:
lm = sm.OLS(brka_excess, exp_var).fit()
lm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   BRKA   R-squared:                       0.290
Model:                            OLS   Adj. R-squared:                  0.282
Method:                 Least Squares   F-statistic:                     36.06
Date:                Wed, 18 Feb 2026   Prob (F-statistic):           1.41e-19
Time:                        13:06:51   Log-Likelihood:                 412.09
No. Observations:                 269   AIC:                            -816.2
Df Residuals:                     265   BIC:                            -801.8
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Mkt-RF         0.6761      0.074      9.155      0.000       0.531       0.821
Constant       0.0055      0.003      1.679      0.094      -0.001       0.012
Value          0.3814      0.109      3.508      0.001       0.167       0.595
Size          -0.5023      0.101     -4.962      0.000      -0.702      -0.303
==============================================================================
Omnibus:                       42.261   Durbin-Watson:                   2.146
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               67.954
Skew:                           0.904   Prob(JB):                     1.75e-15
Kurtosis:                       4.671   Cond. No.                         37.2
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

#### Note: The Value and Size coefficients indicate factor exposure—positive Value suggests BRK.A captures the value premium, while negative Size implies BRK.A tilts toward large-cap firms and does not capture the size premium.

### The Fama-French Benchmark Interpretation

The alpha has fallen from .61% to about 0.55% per month. The loading on the market has moved up from 0.54 to 0.67, which means that adding these new explanatory factors did change things. If we had added irrelevant variables, the loading on the market would be unaffected.

We can interpret the loadings on Value being positive as saying that Hathaway has a significant Value tilt - which should not be a shock to anyone that follows Buffet. Additionally, the negative tilt on size suggests that Hathaway tends to invest in large companies, not small companies.

In other words, Hathaway appears to be a Large Value investor. Of course, you knew this if you followed the company, but the point here is that numbers reveal it!

The new way to interpret each dollar invested in Hathaway is: 67 cents in the market, 33 cents in Bills, 38 cents in Value stocks and short 38 cents in Growth stocks, short 50 cents in SmallCap stocks and long 50 cents in LargeCap stocks. If you did all this, you would still end up underperforming Hathaway by about 55 basis points per month.

#### Check whether Buffett’s tilts remain consistent through 2018.

In [34]:
brka_excess = brka_m['2012':'2018'] - fff.loc['2012':'2018', ['RF']].values
mkt_excess = fff.loc['2012':'2018', ['Mkt-RF']]
exp_var = mkt_excess.copy()
lm = rk.regress(brka_excess, exp_var, alpha = True)
lm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   BRKA   R-squared:                       0.477
Model:                            OLS   Adj. R-squared:                  0.471
Method:                 Least Squares   F-statistic:                     74.93
Date:                Wed, 18 Feb 2026   Prob (F-statistic):           3.48e-13
Time:                        13:26:45   Log-Likelihood:                 185.91
No. Observations:                  84   AIC:                            -367.8
Df Residuals:                      82   BIC:                            -363.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Mkt-RF         0.7952      0.092      8.656      0.000       0.612       0.978
Alpha          0.0039      0.003      1.284      0.203      -0.002       0.010
==============================================================================
Omnibus:                        2.722   Durbin-Watson:                   1.819
Prob(Omnibus):                  0.256   Jarque-Bera (JB):                2.635
Skew:                           0.374   Prob(JB):                        0.268
Kurtosis:                       2.559   Cond. No.                         31.4
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [35]:
exp_var["Value"] = fff.loc['2012':'2018', ['HML']]
exp_var["Size"] = fff.loc['2012':'2018', ['SMB']]
lm = rk.regress(brka_excess, exp_var, alpha = True)
lm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   BRKA   R-squared:                       0.590
Model:                            OLS   Adj. R-squared:                  0.575
Method:                 Least Squares   F-statistic:                     38.45
Date:                Wed, 18 Feb 2026   Prob (F-statistic):           1.73e-15
Time:                        13:29:11   Log-Likelihood:                 196.15
No. Observations:                  84   AIC:                            -384.3
Df Residuals:                      80   BIC:                            -374.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Mkt-RF         0.8673      0.085     10.168      0.000       0.698       1.037
Value          0.4487      0.120      3.727      0.000       0.209       0.688
Size          -0.3658      0.119     -3.083      0.003      -0.602      -0.130
Alpha          0.0032      0.003      1.162      0.249      -0.002       0.009
==============================================================================
Omnibus:                        1.491   Durbin-Watson:                   2.062
Prob(Omnibus):                  0.474   Jarque-Bera (JB):                1.514
Skew:                           0.255   Prob(JB):                        0.469
Kurtosis:                       2.584   Cond. No.                         47.8
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""